
# Handling GitHub Invalid Notebooks
_Last updated: 21/4/25_

## The Problem

When trying to view Jupyter notebooks on GitHub, we sometimes see an **“Invalid Notebook”** message:

```text
Invalid Notebook

There was an error rendering your Notebook: the 'state' key is missing from 'metadata.widgets'.  
Add 'state' to each, or remove 'metadata.widgets'.
```

![Screenshot of GitHub error](https://gist.github.com/user-attachments/assets/baacf996-4594-45b5-8d8d-0a21e3db37dd)

This often happens with notebooks created in *Google Colab* because Colab structures notebook metadata differently than what GitHub’s renderer expects.

## Jupyter Widgets

[Jupyter Widgets](https://ipywidgets.readthedocs.io/en/latest) are interactive browser controls for Jupyter notebooks—sliders, accordions, maps, etc.  
The notebook’s JSON includes the **state** and **version** of each widget.

## Solutions (workarounds)

Below are several workarounds. I’ll update this guide if I (or anyone) finds a more complete solution.

### 1. Add an empty `state`

Some people [report](https://github.com/orgs/community/discussions/155944#discussioncomment-12898793) that simply adding an empty state field to the `widgets` metadata works:

```json
"metadata": {
  "widgets": {
    "state": {}
  }
}
```

This is loss‑less, so it’s always worth trying first.

### 2. Clear cell outputs

Another [workaround](https://github.com/orgs/community/discussions/155944#discussioncomment-12749143) is to clear the outputs of cells that use widgets.  
This is the quickest fix if you know which cells are the culprits and you don’t mind deleting their outputs.

### 3. Delete the entire `widgets` field

A more drastic—but effective—option is to delete `"widgets"` from the notebook JSON altogether.

> ⚠️ **Caveat:** You’ll lose widget state (e.g., slider positions) and their binding to outputs. When you re‑run the notebook the widgets will be recreated, but any saved state is gone.

### How‑to (CLI)

[Source](https://github.com/orgs/community/discussions/155944#discussioncomment-12845735)

```sh
# Backup (recommended)
cp notebook.ipynb notebook_backup.ipynb

# Remove the widgets metadata
jq 'del(.metadata.widgets)' notebook.ipynb > temp.ipynb

# Replace the original
mv temp.ipynb notebook.ipynb
```

### How‑to (Python)

```python
import os, json, sys, pathlib

def clean_notebook(path: str) -> bool:
    path = pathlib.Path(path).expanduser().resolve()
    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    if nb.get("metadata", {}).pop("widgets", None) is not None:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(nb, f, indent=2)
        print(f"Cleaned {path}")
        return True
    return False

if __name__ == "__main__":
    for p in sys.argv[1:]:
        clean_notebook(p)
```

### 4. Convert to HTML

As a fallback you can render the notebook to HTML with `nbconvert`—useful for sharing a static view:

```python
import subprocess
subprocess.run(["jupyter", "nbconvert", "your_notebook.ipynb", "--to", "html"])
```

### 5. Automate with Git hooks / GitHub Actions

You can embed the Python cleaner above into a **pre‑commit hook** so every notebook pushed to GitHub is auto‑fixed.

#### Example `pre-commit` hook

```bash
#!/usr/bin/env bash
set -e

staged=$(git diff --cached --name-only --diff-filter=ACM | grep '\.ipynb$' | grep -v '.ipynb_checkpoints') || true
[[ -z "$staged" ]] && exit 0

python - <<'PY'
import json, sys, pathlib
for p in sys.argv[1:]:
    nb = json.loads(pathlib.Path(p).read_text(encoding="utf-8"))
    nb.get("metadata", {}).pop("widgets", None)
    pathlib.Path(p).write_text(json.dumps(nb, indent=2), encoding="utf-8")
PY $staged

git add $staged
```

---

**Happy notebook hacking!**  
Feel free to adapt these snippets to your workflow.
